# Step 2 – Fit the ML models

For each seed, fold and ROI: the fold is the test set, the other 4 folds are the training set. ROI slide vectors are joined on ID (`block_name + "_01"`) and 7 models are fitted with default scikit-learn settings: `log`, `knn`, `mlp`, `svm`, `dtr`, `xgb`, `rfo`.

Output: `<RESULTS_DIR>/<magnitude>/<group>_s<seed>_<ROI>/tst_part<k>.csv`

Run after Step 1. Set the sample paths in the first code cell to your own data locations before running.

In [ ]:
import glob
import os
import warnings
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

import common_mlops_lib as clm
import commonfilelib as cfl

warnings.filterwarnings("ignore")  # numpy matmul RuntimeWarnings from scikit-learn on macOS

# ---- User input ----
VECTORS_ROOT = Path("/path/to/BMI_DATA/VECTORS")   # folder containing Demographics_321/   (SAMPLE PATH)
DATAPACK_DIR = "/path/to/BMI_DATA/datapack"         # partition CSVs from Step 1         (SAMPLE PATH)
RESULTS_DIR = "/path/to/BMI_DATA/results"           # per-fold results are written here  (SAMPLE PATH)
GROUP = "large_group"
SEEDS = ["0011", "35317"]                # zero-padded, as in the datapack file names
PARTITIONS = ["0", "1", "2", "3", "4"]
MAGNITUDE = "20x"

# ROI name -> vector file, relative to <VECTORS_ROOT>/Demographics_321/<MAGNITUDE>/CANVOI_v1_1/
ROI_PATH_MAP = {
    "WSI": "collective_slide_vectors_non_ROI/full_vectors_compressed.h5",
    "Tumor": "collective_slide_vectors_90_ROI/Positive/full_vectors_compressed.h5",
    "NonTumor": "collective_slide_vectors_90_ROI/Negative/full_vectors_compressed.h5",
    "TumorDense": "collective_slide_vectors_90_ROI_cluster/TumorDense/full_vectors_compressed.h5",
    "TumorNotDense": "collective_slide_vectors_90_ROI_cluster/TumorNotDense/full_vectors_compressed.h5",
    "StromaDense": "collective_slide_vectors_90_ROI_cluster/StromaDense/full_vectors_compressed.h5",
    "StromaNotDense": "collective_slide_vectors_90_ROI_cluster/StromaNotDense/full_vectors_compressed.h5",
}
ROIS = list(ROI_PATH_MAP)                # or a subset, e.g. ["TumorDense", "StromaDense"]
# --------------------

base_path = VECTORS_ROOT / f"Demographics_321/{MAGNITUDE}/CANVOI_v1_1"
if not base_path.exists():
    raise FileNotFoundError(f"Vectors not found at {base_path}")

In [ ]:
configs = [(seed, part, roi) for seed in SEEDS for part in PARTITIONS for roi in ROIS]

for seed, part, roi in tqdm(configs, desc="Running experiments"):
    pattern = os.path.join(DATAPACK_DIR, f"{GROUP}_bmi_s{seed}_part*_datapack.csv")
    tst_name = f"{GROUP}_bmi_s{seed}_part{part}_datapack.csv"
    tst_path = os.path.join(DATAPACK_DIR, tst_name)
    trn_files = [f for f in glob.glob(pattern) if not f.endswith(tst_name)]

    if not trn_files or not os.path.exists(tst_path):
        tqdm.write(f"[MISSING] seed={seed}, part={part}, ROI={roi}")
        continue

    trn = pd.concat([pd.read_csv(f) for f in trn_files], ignore_index=True)
    tst = pd.read_csv(tst_path)
    for df in (trn, tst):
        df["ID"] = df["block_name"] + "_01"
        df["BMI_High"] = (df["status"] == "high").astype(int)

    if set(trn["ID"]) & set(tst["ID"]):
        tqdm.write(f"[LEAK] train/test overlap: seed={seed}, part={part}")
        continue

    all_vecs = cfl.get_vectors_from_h5(str(base_path / ROI_PATH_MAP[roi]))

    # Inner join: keep only slides that have a vector in this ROI
    trn_df = trn.merge(all_vecs, on="ID", how="inner")
    tst_df = tst.merge(all_vecs, on="ID", how="inner")
    if len(trn_df) == 0 or len(tst_df) == 0:
        tqdm.write(f"[MISSING VECTORS] seed={seed}, part={part}, ROI={roi}")
        continue

    drop_cols = ["ID", "status", "block_name", "BMI_High", "index"]
    res_df, _, _ = clm.build_ml_models3(
        Xtrn=trn_df.drop(columns=drop_cols, errors="ignore"),
        ytrn=trn_df["BMI_High"],
        Xtst=tst_df.drop(columns=drop_cols, errors="ignore"),
        ytst=tst_df["BMI_High"],
        verbose=False,
        grid_search=False,
    )

    out_dir = Path(RESULTS_DIR) / MAGNITUDE / f"{GROUP}_s{seed}_{roi}"
    out_dir.mkdir(parents=True, exist_ok=True)
    res_df.to_csv(out_dir / f"tst_part{part}.csv", index=False)

Example of one result file (AUC, accuracy, MCC, F1, specificity, sensitivity and confusion-matrix counts per model):

In [ ]:
pd.read_csv(Path(RESULTS_DIR) / MAGNITUDE / f"{GROUP}_s{SEEDS[0]}_{ROIS[0]}" / "tst_part0.csv")